# Preprocess CMIP6 data
Script for downloading and saving CMIP6 files, with ability to subset by time and space. CMIP6 data is lazily loaded directly from the cloud, using the Pangeo - Google Cloud Public Dataset Program collaboration (more info [here](https://medium.com/pangeo/cmip6-in-the-cloud-five-ways-96b177abe396)).

# Setup

In [3]:
import xarray as xr
import xagg as xa
import pandas as pd
import numpy as np
import cftime
from tqdm.notebook import tqdm
import re
import copy
from operator import itemgetter
import gcsfs
import os
import yaml
import warnings

from funcs_support import get_varlist,get_params,get_filepaths
from funcs_aux import _load_gwls

In [3]:
dir_list = get_params()

In [4]:
# Set whether to regrid 360-day calendars to 365-day calendars
# (probably don't do this while saving files. Only do this in 
# processing code that doesn't affect the original file)
regrid_360 = False

In [5]:
# Set whether to only download stuff that is in gwl_info for ssp245
subset_to_gwl245 = False

# Set whether to only download stuff that already has data for a specific experiment
exp_to_subset = 'historical'
subset_to_exp = True

# Set whether to make the start year earlier if historical and the desired start year
# is past the desired GWL
overwrite_startyear_bygwlinfo = True

In [6]:
# Set how many runs max to download
#max_nruns = 30
max_nruns = 100

In [17]:
# Set parameters for spatiotemporal subsetting
subset_params_all = [{'time' : {'historical' : ['1958-01-01','2014-12-31'],
                                'amip' : ['1958-01-01','2014-12-31'],
                              'ssp245' : ['2015-01-01','2099-12-31'],
                                'ssp370' : ['2015-01-01','2099-12-31'],
                              'ssp585' : ['2015-01-01','2099-12-31']},
                      'lat' : [-57,85],'lon' : [-180,180], 'fn_suffix' : '', # covers all inhabited land
                      'lon_range' : 180,'lon_origin' : -180}]

#data_params_all = [{'experiment_id':'historical','table_id':'day','variable_id':'pr'},
#                   {'experiment_id':'ssp370','table_id':'day','variable_id':'pr'},
#                   {'experiment_id':'ssp585','table_id':'day','variable_id':'pr'}]

mods = ['ACCESS-ESM1-5','CNRM-CM6-1','CNRM-ESM2-1','EC-Earth3-Veg','CESM2-WACCM','FGOALS-g3',
        'CanESM5', 'EC-Earth3', 'IPSL-CM6A-LR', 'MIROC6','MPI-ESM1-2-LR','MRI-ESM2-0',
        'GISS-E2-1-G']

var = 'tasmax'
#var = 'tas'

data_params_all = [[{'experiment_id':'historical','table_id':'day','variable_id':var,'source_id':mod},
                  {'experiment_id':'ssp585','table_id':'day','variable_id':var,'source_id':mod},
                  {'experiment_id':'ssp370','table_id':'day','variable_id':var,'source_id':mod}, 
                   {'experiment_id':'ssp245','table_id':'day','variable_id':var,'source_id':mod}]
                  for mod in mods]
data_params_all = [x for xs in data_params_all for x in xs]


In [19]:
# Get GWL info
gwl_info = _load_gwls()

# Process

In [20]:
## Prepare the full query for all the datasets that will end up getting use in this 
# process - this is to create the master dataset, so to build up the 'model' and 
# 'experiment' dimension in the dataset with all the values that will end up used
source_calls = np.zeros(len(data_params_all[0].keys()))

for key in data_params_all[0].keys():
    if len(np.unique([x[key] for x in data_params_all]))==1:
        source_calls[list(data_params_all[0].keys()).index(key)] = 1
        
# To skip the 'other' one in the join below, hopefully it works.


# First get all the ones with the same value for each key 
subset_query = ' and '.join([k+" == '"+data_params_all[0][k]+"'" for k in itemgetter(*source_calls.nonzero()[0])(list(data_params_all[0].keys())) if k != 'other'])

# Now add all that are different between subset params - i.e. those that need an OR statement
# These have to be in two statements, because if there's only one OR'ed statement, then the 
# for k in statement goes through the letters instead of the keys. 
if len((source_calls-1).nonzero()[0])==1:
    subset_query=subset_query+' and ('+') and ('.join([' or '.join([k+" == '"+data_params[k]+"'" for data_params in data_params_all]) 
              for k in [itemgetter(*(source_calls-1).nonzero()[0])(list(data_params_all[0].keys()))] if k != 'other'])+')'
elif len((source_calls-1).nonzero()[0])>1:
    subset_query=subset_query+' and ('+') and ('.join([' or '.join([k+" == '"+data_params[k]+"'" for data_params in data_params_all]) 
              for k in itemgetter(*(source_calls-1).nonzero()[0])(list(data_params_all[0].keys())) if k != 'other'])+')'


In [21]:
# Access google cloud storage links
fs = gcsfs.GCSFileSystem(token='anon', access='read_only')
# Get info about CMIP6 datasets
cmip6_datasets = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
# Get subset based on the data params above (for all search parameters)
cmip6_sub = cmip6_datasets.query(subset_query)

if subset_to_gwl245:
    cmip6_sub = cmip6_sub.loc[[(row[1]['source_id'],row[1]['member_id']) in gwl_info.index for row in cmip6_sub.iterrows()],:]

if len(cmip6_sub) == 0:
    warnings.warn('Query unsuccessful, no files found! Check to make sure your table_id matches the domain - SSTs are listed as "Oday" instead of "day" for example')
    

In [22]:
cmip6_sub

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
37719,CMIP,CNRM-CERFACS,CNRM-CM6-1,historical,r1i1p1f2,day,tasmax,gr,gs://cmip6/CMIP6/CMIP/CNRM-CERFACS/CNRM-CM6-1/...,NaN,20180917
40064,CMIP,NASA-GISS,GISS-E2-1-G,historical,r1i1p1f1,day,tasmax,gn,gs://cmip6/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/hi...,NaN,20181015
43366,CMIP,CNRM-CERFACS,CNRM-CM6-1,historical,r2i1p1f2,day,tasmax,gr,gs://cmip6/CMIP6/CMIP/CNRM-CERFACS/CNRM-CM6-1/...,NaN,20181126
44206,CMIP,CNRM-CERFACS,CNRM-ESM2-1,historical,r1i1p1f2,day,tasmax,gr,gs://cmip6/CMIP6/CMIP/CNRM-CERFACS/CNRM-ESM2-1...,NaN,20181206
50457,CMIP,CNRM-CERFACS,CNRM-CM6-1,historical,r8i1p1f2,day,tasmax,gr,gs://cmip6/CMIP6/CMIP/CNRM-CERFACS/CNRM-CM6-1/...,NaN,20190125
...,...,...,...,...,...,...,...,...,...,...,...
522989,ScenarioMIP,MRI,MRI-ESM2-0,ssp245,r2i1p1f1,day,tasmax,gn,gs://cmip6/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ss...,NaN,20210830
523010,ScenarioMIP,MRI,MRI-ESM2-0,ssp245,r4i1p1f1,day,tasmax,gn,gs://cmip6/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ss...,NaN,20210830
523014,ScenarioMIP,MRI,MRI-ESM2-0,ssp245,r3i1p1f1,day,tasmax,gn,gs://cmip6/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ss...,NaN,20210830
523015,ScenarioMIP,MRI,MRI-ESM2-0,ssp245,r5i1p1f1,day,tasmax,gn,gs://cmip6/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ss...,NaN,20210830


In [ ]:
#------ Process by variable and dataset in the subset ------
overwrite=False
for data_params in data_params_all:
    # Get subset based on the data params above, now just for this one variable
    cmip6_sub = cmip6_datasets.query(' and '.join([k+" == '"+data_params[k]+"'" for k in data_params.keys() if k != 'other']))

    if subset_to_exp:
        df = get_filepaths(source_dir = 'raw').query('varname == "'+data_params['variable_id']+'" and '+
                                                      'model == "'+data_params['source_id']+'" and '+
                                                      'freq == "'+data_params['table_id']+'" and '+
                                                      'exp == "'+exp_to_subset+'"')

    
    if len(cmip6_sub)>0:
        # Subset to just the grid label that shows up the most 
        # in the subset list
        grid_labels, grid_counts = np.unique(np.concatenate((cmip6_sub.groupby(['source_id','experiment_id','member_id','variable_id','table_id']).
                                                             apply(lambda x: np.unique(x.grid_label),
                                                                   include_groups=False)).values),return_counts = True)
        
        grid_label = grid_labels[np.argmax(grid_counts)]
    
        # Subset files to only those with that grid label
        cmip6_sub = cmip6_sub.query('grid_label == "'+grid_label+'"')
    
        # Subset files to only those for which we have GWL infos, if desired
        if subset_to_gwl245:
            cmip6_sub = cmip6_sub.loc[[(row[1]['source_id'],row[1]['member_id']) in gwl_info.index for row in cmip6_sub.iterrows()],:]
    
        # Subset files to only those for which we already have files of a different GWL
        if subset_to_exp:
            if data_params['experiment_id'] != exp_to_subset:
                cmip6_sub = cmip6_sub.loc[[run in df['run'].values for run in cmip6_sub.member_id.values]]
        
        # Subset files to only max nruns 
        if max_nruns is not None:
            cmip6_sub['runid'] = [int(re.split('i',re.split('r',mid)[1])[0]) for mid in cmip6_sub.member_id.values]
            cmip6_sub = cmip6_sub.sort_values('runid').groupby(['source_id','experiment_id','grid_label',]).head(max_nruns)
        
        for url_idx,url in tqdm(enumerate(cmip6_sub.zstore.values),total=len(cmip6_sub)):
            print('processing run '+cmip6_sub.member_id.values[url_idx]+' from '+data_params['variable_id']+' '+
                     data_params['table_id']+' '+url.split('/')[6]+' '+
                     data_params['experiment_id']+'!')
            try:
                output_fns = [None]*len(subset_params_all)
                path_exists = [None]*len(subset_params_all)
                for idx,subset_params in enumerate(subset_params_all):
                    # Open dataset metadata
                    ds = xr.open_zarr(fs.get_mapper(url),consolidated=True)

                    #ds = xr.open_zarr(fs.get_mapper(url),consolidated=True,decode_times=xr.coders.CFDatetimeCoder(use_cftime=True))
                    
                    subset_params = copy.deepcopy(subset_params)
                    if 'member_id' in data_params:
                        member_id = data_params['member_id']
                    else:
                        member_id = url.split('/')[8]
                
                    mod = url.split('/')[6]
                    
                    if 'time' in subset_params:
                        if (data_params['experiment_id'] == 'historical') and overwrite_startyear_bygwlinfo:
                            try:
                                gwl_startyear = gwl_info.loc[(mod,member_id)].query('exp == "historical"').sort_values('warming_level',axis=0).iloc[0]['start_year']
                                if int(subset_params['time'][data_params['experiment_id']][0][0:4]) > gwl_startyear:
                                    subset_params['time'][data_params['experiment_id']] = [str(gwl_startyear)+'-01-01',subset_params['time'][data_params['experiment_id']][1]]
                            except:
                                pass
                        
                        time_str = '-'.join([re.sub('-','',t) for t in subset_params['time'][data_params['experiment_id']]])
                    else:
                        time_str = ''

                    # REALLY SHOULD SPIN THIS OFF INTO A SEPARATE FUNCTION THAT DOES SUBSETTING BY TIME
                    # AND CHECKS BOTH START AND FINISH....... 
                    try:
                        if pd.to_datetime(str(ds.time.max().values)[0:10]) < pd.to_datetime(time_str[9:None]):
                            time_str = time_str[0:9]+re.sub(r'\-','',str(ds.time.max().values)[0:10])
                            warnings.warn('Dataset ends before end of time subset_params... adjusting timestring to '+time_str)
                    except Exception as e:
                        print(e)
                        pass # This is almost certainly not an issue, because that means it's >> 2100
                    output_fns[idx] = (dir_list['raw']+mod+'/'+
                                                                         data_params['variable_id']+'_'+
                                                                         data_params['table_id']+'_'+mod+'_'+
                                                                         data_params['experiment_id']+'_'+member_id+'_'+
                                                                          grid_label+'_'+
                                                                         time_str+
                                                                         subset_params['fn_suffix']+'.nc')
                
                    if 'other' in data_params.keys(): 
                        if 'plev_subset' in data_params['other'].keys():
                            output_fns[idx] = re.sub(data_params['variable_id'],
                                                                                    data_params['other']['plev_subset']['outputfn'],
                                                                                   output_fns[idx])
                
                
                
                    path_exists[idx] = os.path.exists(output_fns[idx])

                if (not overwrite) & all(path_exists):
                    print('All files already created for '+data_params['variable_id']+' '+
                                                                         data_params['table_id']+' '+url.split('/')[6]+' '+
                                                                         data_params['experiment_id']+' '+member_id+', skipped.')
                    continue
                elif any(path_exists):
                    if overwrite:
                        for idx,subset_params in enumerate(subset_params_all):
                            if path_exists[subset_params_all.index(subset_params)]:
                                os.remove(output_fns[idx])
                                warnings.warn('All files already exist for '+data_params['variable_id']+' '+
                                                                                     data_params['table_id']+' '+url.split('/')[6]+' '+
                                                                                     data_params['experiment_id']+' '+member_id+
                                              ', because OVERWRITE=TRUE these files have been deleted.')
    
    
                # Rename to lat / lon (let's hope there's no 
                # Latitude / latitude_1 / etc. in this dataset)
                try:
                    ds = ds.rename({'longitude':'lon','latitude':'lat'})
                except: 
                    pass
    
                # same with 'nav_lat' and 'nav_lon' ???
                try:
                    ds = ds.rename({'nav_lon':'lon','nav_lat':'lat'})
                except: 
                    pass
    
                # If precip, kg/m^2/s, switch to mm/day
                #if data_params['variable_id']=='pr':
                #    ds[data_params['variable_id']] = ds[data_params['variable_id']]*60*60*24
    
                # Fix coordinate doubling (this was an issue in NorCPM1, 
                # where thankfully the values of the variables were nans,
                # though I still don't know how this happened - some lat
                # values were doubled within floating point errors)
                if 'lat' in ds[data_params['variable_id']].sizes:
                    if len(np.unique(np.round(ds.lat.values,10))) != ds.sizes['lat']:
                        ds = ds.isel(lat=(~np.isnan(ds.isel(lon=1,time=1)[data_params['variable_id']].values)).nonzero()[0],drop=True)
                        warnings.warn('Model '+ds.source_id+' has duplicate lat values; attempting to compensate by dropping lat values that are nan in the main variable in the first timestep')
                    if len(np.unique(np.round(ds.lon.values,10))) != ds.sizes['lon']:
                        ds = ds.isel(lon=(~np.isnan(ds.isel(lat=1,time=1)[data_params['variable_id']].values)).nonzero()[0],drop=True)
                        warnings.warn('Model '+ds.source_id+' has duplicate lon values; attempting to compensate by dropping lon values that are nan in the main variable in the first timestep')
    
                # Sort by time, if not sorted (this happened with
                # a model; keeping a warning, cuz this seems weird)
                if 'time' in subset_params:
                    if (ds.time.values != np.sort(ds.time)).any():
                        warnings.warn('Model '+ds.source_id+' has an unsorted time dimension.')
                        ds = ds.sortby('time')
    
                # If 360-day calendar, regrid to 365-day calendar
                if regrid_360:
                    if ds.sizes['dayofyear'] == 360:
                        # Have to put in the compute() because these 
                        # are by default dask arrays, chunked along
                        # the time dimension, and can't interpolate
                        # across dask chunks... 
                        ds = ds.compute().interp(dayofyear=(np.arange(1,366)/365)*360)
                        # And reset it to 1:365 indexing on day of year
                        ds['dayofyear'] = np.arange(1,366)
                        # Throw in a warning, too, why not
                        warnings.warn('Model '+ds.source_id+' has a 360-day calendar; daily values were interpolated to a 365-day calendar')
    
                # Now, save by the subsets desired in subset_params_all above
                for idx,subset_params in enumerate(subset_params_all):
                    subset_params = copy.deepcopy(subset_params)
                    # Make sure this file hasn't already been processed
                    if (not overwrite) & path_exists[idx]:
                        warnings.warn(output_fns[idx]+' already exists; skipped.')
                        continue
    
                    # Make sure the target directory exists
                    if not os.path.exists(dir_list['raw']+url.split('/')[6]+'/'):
                        os.mkdir(dir_list['raw']+url.split('/')[6]+'/')
                        warnings.warn('Directory '+dir_list['raw']+url.split('/')[6]+'/'+' created!')
    
                    # Fix longitude (by setting it to either [-180:180] 
                    # or [0:360] as determined by subset_params, and 
                    # to roll them so the correct range is consecutive 
                    # in lon (so if you're looking at the Equatorial 
                    # Pacific, make it 0:360, with the first lon value
                    # at 45E). 
                    if 'lat' in ds[data_params['variable_id']].sizes:
                        ds_tmp = xa.fix_ds(ds,subset_params)
                        # Now, cutoff the values below the 'lon_origin', 
                        # because slice doesn't work if the indices aren't
                        # montonically increasing (and the above changes it
                        # to [lon_origin:360 0:lon_origin]
                        if np.abs(ds_tmp.lon[0]-subset_params['lon_origin'])>5:
                            ds_tmp = ds_tmp.isel(lon=np.arange(0,(ds_tmp.lon // (subset_params['lon_origin']) == 0).values.nonzero()[0][0]))
                    else:
                        ds_tmp = ds.copy()
                        warnings.warn('fix_ds did not work because of the multi-dimensional index')
    
                    # Subset by time as set in subset_params
                    if 'time' in subset_params:
                        if (data_params['experiment_id'] == 'historical') and overwrite_startyear_bygwlinfo:
                            try:
                                gwl_startyear = gwl_info.loc[(mod,member_id)].query('exp == "historical"').sort_values('warming_level',axis=0).iloc[0]['start_year']
                                if int(subset_params['time'][data_params['experiment_id']][0][0:4]) > gwl_startyear:
                                    subset_params['time'][data_params['experiment_id']] = [str(gwl_startyear)+'-01-01',subset_params['time'][data_params['experiment_id']][1]]
                                    print('Moving start year up from '+subset_params['time'][data_params['experiment_id']][0]+' to '+
                                          str(gwl_startyear)+' to match first GWL...')
                            except:
                                pass
                        if (ds.time.max().dt.day==30) | (type(ds.time.values[0]) == cftime._cftime.Datetime360Day): 
                            # (If it's a 360-day calendar, then subsetting to "12-31"
                            # will throw an error; this switches that call to "12-30")
                            # Also checking explicitly for 360day calendar; some monthly 
                            # data is still shown as 360-day even when it's monthly, and will
                            # fail on date ranges with date 31 in a month
                            ds_tmp = (ds_tmp.sel(time=slice(subset_params['time'][data_params['experiment_id']][0],
                                                    re.sub('-31','-30',subset_params['time'][data_params['experiment_id']][1]))))
                        else:
                            ds_tmp = (ds_tmp.sel(time=slice(*subset_params['time'][data_params['experiment_id']])))
    
                    # Subset by space as set in subset_params
                    if 'lat' in subset_params.keys():
                        if not 'lat' in ds[data_params['variable_id']].sizes:
                            ds_tmp = ds_tmp.where((ds_tmp.lat >= subset_params['lat'][0]) & (ds_tmp.lat <= subset_params['lat'][1]) &
                             (ds_tmp.lon >= subset_params['lon'][0]) & (ds_tmp.lon <= subset_params['lon'][1]),drop=True)
                        else:
                            ds_tmp = (ds_tmp.sel(lat=slice(*subset_params['lat']),
                                                 lon=slice(*subset_params['lon'])))
    
                    # If subsetting by pressure level...
                    if 'other' in data_params.keys():
                        if 'plev_subset' in data_params['other'].keys():
                            # Have to use np.allclose for floating point errors
                            try:
                                ds_tmp = ds_tmp.isel(plev=np.where([np.allclose(p,data_params['other']['plev_subset']['plev']) for p in ds_tmp.plev])[0][0])
                                ds_tmp = ds_tmp.rename({data_params['variable_id']:data_params['other']['plev_subset']['outputfn']})
                            except KeyError:
                                print('The pressure levels: ')
                                print(ds_tmp.plev.values)
                                print(' do not contain '+str(data_params['other']['plev_subset']['plev'])+'; skipping.')
                                del ds_tmp
                                continue
                    # Trick to make the loading process go faster (otherwise it
                    # gets stuck forever in .to_netcdf below; and .load() is 
                    # just as slow for some reason)
                    #if 'time' in subset_params:
                        #tmp = ds_tmp.mean('time')
                        #del tmp
    
                    # Save as NetCDF file
                    if ds_tmp.sizes['time']>0:
                        try:
                            ds_tmp.to_netcdf(output_fns[idx])
                        except ValueError:
                            print('issue with export; skipping')
                            #del ds_tmp
                            continue
                    else:
                        print('time dimension is 0, skipping')
                        continue
                        
    
                    # Status update
                    print(output_fns[idx]+' processed!')
    
                del ds, ds_tmp, subset_params
            except AssertionError:
                print('checksum error with model '+url.split('/')[6]+', skipping for now.')
                continue
    else:
        print('No files found for '+', '.join([i for k,i in data_params.items()]))
        

  0%|          | 0/29 [00:00<?, ?it/s]

processing run r1i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r1i1p1f1, skipped.
processing run r2i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r2i1p1f1, skipped.
processing run r3i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r3i1p1f1, skipped.
processing run r4i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r4i1p1f1, skipped.
processing run r5i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r5i1p1f1, skipped.
processing run r6i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created for tasmax day ACCESS-ESM1-5 historical r6i1p1f1, skipped.
processing run r8i1p1f1 from tasmax day ACCESS-ESM1-5 historical!
All files already created fo

  0%|          | 0/30 [00:00<?, ?it/s]

processing run r1i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r1i1p1f2, skipped.
processing run r2i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r2i1p1f2, skipped.
processing run r3i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r3i1p1f2, skipped.
processing run r4i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r4i1p1f2, skipped.
processing run r5i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r5i1p1f2, skipped.
processing run r6i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r6i1p1f2, skipped.
processing run r7i1p1f2 from tasmax day CNRM-CM6-1 historical!
All files already created for tasmax day CNRM-CM6-1 historical r7i1

  0%|          | 0/1 [00:00<?, ?it/s]

...

All files already created for tasmax day EC-Earth3 historical r13i1p1f1, skipped.
processing run r14i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r14i1p1f1, skipped.
processing run r15i1p1f1 from tasmax day EC-Earth3 historical!


/glade/derecho/scratch/schwarzwald/tmp/ipykernel_18929/982438669.py:79: UserWarning: Dataset ends before end of time subset_params... adjusting timestring to 19470101-20141230
  warnings.warn('Dataset ends before end of time subset_params... adjusting timestring to '+time_str)


All files already created for tasmax day EC-Earth3 historical r15i1p1f1, skipped.
processing run r16i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r16i1p1f1, skipped.
processing run r17i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r17i1p1f1, skipped.
processing run r18i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r18i1p1f1, skipped.
processing run r19i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r19i1p1f1, skipped.
processing run r21i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r21i1p1f1, skipped.
processing run r22i1p1f1 from tasmax day EC-Earth3 historical!
All files already created for tasmax day EC-Earth3 historical r22i1p1f1, skipped.
processing run r23i1p1f1 from tasmax day EC-Eart

  0%|          | 0/30 [00:00<?, ?it/s]

processing run r1i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r1i1p1f1, skipped.
processing run r4i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r4i1p1f1, skipped.
processing run r6i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r6i1p1f1, skipped.
processing run r7i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r7i1p1f1, skipped.
processing run r9i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r9i1p1f1, skipped.
processing run r10i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r10i1p1f1, skipped.
processing run r11i1p1f1 from tasmax day EC-Earth3 ssp245!
All files already created for tasmax day EC-Earth3 ssp245 r11i1p1f1, skipped.
processing run r12i1p1f1 from tasmax day EC-Earth3 